In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# 월별 CSV 파일 3개 불러오기

monthly_1 = pd.read_csv(
    "../data/raw(원본)/부산항_2020_2021_월별_분석용.csv",
    encoding="utf-8-sig"
)

monthly_2 = pd.read_csv(
    "../data/raw(원본)/부산항_2022_2023_월별_분석용.csv",
    encoding="utf-8-sig"
)

monthly_3 = pd.read_csv(
    "../data/raw(원본)/부산항_2024_2025_월별_분석용.csv",
    encoding="utf-8-sig"
)

# 세 파일 합치기
monthly_all = pd.concat(
    [monthly_1, monthly_2, monthly_3],
    ignore_index=True
)

monthly_all.shape

(45122, 11)

In [3]:
# 해당 월에 물동량이 없는 경우 0으로 처리
monthly_all[
    ['total_teu', 'laden_teu', 'empty_teu']
] = monthly_all[
    ['total_teu', 'laden_teu', 'empty_teu']
].fillna(0)

In [4]:
monthly_import = (
    monthly_all[
        monthly_all['flow'] == '입항'
    ]
    .groupby(
        ['year', 'month', 'country']
    )['total_teu']
    .sum()
    .reset_index()
)

monthly_import = monthly_import.rename(
    columns={'total_teu': '수입'}
)

In [5]:
monthly_export = (
    monthly_all[
        monthly_all['flow'] == '출항'
    ]
    .groupby(
        ['year', 'month', 'country']
    )['total_teu']
    .sum()
    .reset_index()
)

monthly_export = monthly_export.rename(
    columns={'total_teu': '수출'}
)

In [6]:
monthly_trans = (
    monthly_all[
        monthly_all['flow'].isin(
            ['입항환적', '출항환적']
        )
    ]
    .groupby(
        ['year', 'month', 'country']
    )['total_teu']
    .sum()
    .reset_index()
)

monthly_trans = monthly_trans.rename(
    columns={'total_teu': '환적'}
)

In [7]:
monthly_master = pd.merge(
    monthly_import,
    monthly_export,
    on=['year', 'month', 'country'],
    how='outer'
)

monthly_master = pd.merge(
    monthly_master,
    monthly_trans,
    on=['year', 'month', 'country'],
    how='outer'
)

In [8]:
monthly_master[
    ['수입', '수출', '환적']
] = monthly_master[
    ['수입', '수출', '환적']
].fillna(0)

monthly_master['전체물동량'] = (
    monthly_master['수입']
    + monthly_master['수출']
    + monthly_master['환적']
)

In [9]:
monthly_master = monthly_master.rename(
    columns={
        'year': '연도',
        'month': '월',
        'country': '국가명'
    }
)

monthly_master = monthly_master.sort_values(
    ['연도', '월', '국가명']
).reset_index(drop=True)

monthly_master.head(10)

,연도,월,국가명,수입,수출,환적,전체물동량
0,2020,1,가나,24.00,176.50,65.0,265.5
1,2020,1,가봉,0.00,0.00,4.0,4.0
2,2020,1,가이아나,24.00,1.00,515.0,540.0
3,2020,1,감비아,11.00,1.00,12.0,24.0
4,2020,1,과들루프,0.00,0.00,11.0,11.0
5,2020,1,과테말라,430.00,181.00,1622.5,2233.5
6,2020,1,괌,841.25,150.75,368.5,1360.5
7,2020,1,그레나다,0.00,1.00,29.0,30.0
8,2020,1,그리스,1156.00,1129.00,1430.0,3715.0
9,2020,1,기니,68.00,7.00,2.0,77.0


In [10]:
monthly_master.groupby(
    '연도'
)['전체물동량'].sum()

연도
2020    21823961.00
2021    22706123.75
2022    22076784.50
2023    23152998.00
2024    24402020.00
2025    24882345.00
Name: 전체물동량, dtype: float64

In [11]:
monthly_master.to_csv(
    "../data/processed(정제)/06_monthly_master_2020_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

이건 월별 데이터


In [12]:
top10_total = (
    monthly_all
    .groupby('country')['total_teu']
    .sum()
    .reset_index()
    .sort_values('total_teu', ascending=False)
    .head(10)
)

top10_total

,country,total_teu
141,중국,38355133.25
55,미국,21518720.75
137,일본,17397957.25
46,멕시코,4474665.50
152,캐나다,4272509.75
67,베트남,4137722.00
33,러시아,3517364.25
24,대한민국,2952596.50
146,칠레,2513106.25
136,인도네시아,2494232.75


In [13]:
# 전체 물동량
all_total = monthly_all['total_teu'].sum()

# 전체 대비 비중 계산
top10_total['비중(%)'] = (
    top10_total['total_teu']
    / all_total
    * 100
)

# 순위 추가
top10_total['순위'] = range(1, 11)

# 보기 좋은 순서로 정리
top10_total = top10_total[
    ['순위', 'country', 'total_teu', '비중(%)']
]

top10_total

,순위,country,total_teu,비중(%)
141,1,중국,38355133.25,27.584843
55,2,미국,21518720.75,15.476169
137,3,일본,17397957.25,12.512534
46,4,멕시코,4474665.50,3.218160
152,5,캐나다,4272509.75,3.072770
67,6,베트남,4137722.00,2.975831
33,7,러시아,3517364.25,2.529673
24,8,대한민국,2952596.50,2.123494
146,9,칠레,2513106.25,1.807415
136,10,인도네시아,2494232.75,1.793841
